# ✈️ HOROv1 — Pipeline de Análise de Ventos para Aeródromos

**RBAC154 / ICAO Annex 14 — Fator de Operação (FO) e Rosa dos Ventos**

Desenvolvido por **John Heberty de Freitas** — john.7heberty@gmail.com

---

## 📋 O que este notebook faz

| Seção | Descrição |
|-------|-----------|
| **1 — Ambiente** | Instalação de dependências e configuração do caminho do projeto |
| **2 — Configuração** | Editor visual do `config_runway.json` — comprimento de pista, bandas de vento, cores, fps, etc. |
| **3 — Dados de Entrada** | Verificação e prévia dos CSVs em `data/raw/` |
| **4 — Pipeline Completo** | Execução de todos os estágios com um clique |
| **5 — Estágio a Estágio** | Execução individual para depuração |
| **6 — Resultados** | Visualização do MP4, GIF, PNG e JSON |

---

## 📂 Fluxo dos Dados

```
data/raw/ (CSV)  →  Bronze  →  Silver  →  Gold  →  data/gold/exports/
                 ingest   transform  analyze   MP4 + GIF + PNG
```

> 💡 **Dica:** Execute as seções em ordem para um pipeline completo.  
> Para re-executar rapidamente após editar o `config_runway.json`, use somente a **Seção 4**.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SEÇÃO 1 — AMBIENTE
# Instala/verifica dependências. Execute uma vez por ambiente.
# ─────────────────────────────────────────────────────────────────────────────
%pip install -q pandas numpy windrose matplotlib scikit-learn opencv-python \
             selenium webdriver-manager Unidecode chardet python-dateutil \
             fastparquet pyarrow structlog pillow

print("✅ Dependências verificadas.")


In [ ]:
import os, sys, json, subprocess, textwrap
from pathlib import Path
from IPython.display import display, Image, Video, JSON, Markdown

# Garante que a raiz do projeto está no sys.path
PROJECT_ROOT = Path(os.getcwd()).resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Importa configuração e logger
from pipeline.core.config import PipelineConfig, cfg
from pipeline.core.logger import configure_logging, get_logger
from pipeline.core.models import PipelineContext

configure_logging(level="INFO")
cfg.ensure_dirs()

print(f"✅ Projeto: {PROJECT_ROOT}")
print(f"   data/raw    → {cfg.output.data_raw}")
print(f"   data/silver → {cfg.output.data_silver}")
print(f"   data/gold   → {cfg.output.data_gold}")


---
## ⚙️ SEÇÃO 2 — Configuração do Aeródromo

Edite os valores abaixo e execute a célula **"💾 Salvar config_runway.json"** para aplicar.  
As alterações serão gravadas em `config_runway.json` na raiz do projeto.

> ℹ️ Todos os parâmetros têm comentários explicativos. Altere apenas os que precisar.


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║          EDITE OS VALORES ABAIXO CONFORME SEU AERÓDROMO                ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# ─── PISTA ────────────────────────────────────────────────────────────────────
# Comprimento da pista de referência em metros.
# Determina o limite de vento cruzado conforme RBAC154:
#   ≥ 1500 m → 20 kt | 1200–1500 m → 13 kt | < 1200 m → 10 kt
RUNWAY_LENGTH_M = 1199

# ─── LOCALIZAÇÃO ──────────────────────────────────────────────────────────────
# Coordenadas do aeródromo em graus decimais.
# Se fornecidas, substituem as coordenadas lidas do CSV.
# Use None para desativar (o sistema lê do CSV).
LATITUDE  = -23.428375   # Sul  = negativo
LONGITUDE = -46.467887   # Oeste = negativo

# ─── DECLINAÇÃO MAGNÉTICA ─────────────────────────────────────────────────────
# None = consulta automática ao NOAA (requer Google Chrome instalado)
# Número float = usa o valor fixo (ex.: -21.95 = 21.95° Oeste)
DECLINATION = None

# ─── BANDAS DE VELOCIDADE DO VENTO (nós) ─────────────────────────────────────
# Lista de 5 limites cria 6 faixas:
#   [0-3]  [3-13]  [13-20]  [20-25]  [25-40]  [40+]
WIND_BANDS_KTS = [3, 13, 20, 25, 40]

# ─── CORES DA ROSA DOS VENTOS (PNG) ──────────────────────────────────────────
# Formato [R, G, B] 0–255. Uma cor por banda (mesma ordem acima).
WIND_COLORS_RGB = [
    [180, 180, 180],   # banda 0: calmarias  (cinza)
    [ 30, 100, 255],   # banda 1:  3–13 kt   (azul)
    [ 40, 255,  40],   # banda 2: 13–20 kt   (verde)
    [255, 255,   0],   # banda 3: 20–25 kt   (amarelo)
    [255, 150,   0],   # banda 4: 25–40 kt   (laranja)
    [255,  30,   0],   # banda 5:   40+ kt   (vermelho)
]

# ─── CORES DO VÍDEO/GIF (podem ser diferentes da rosa) ───────────────────────
VIDEO_COLORS_RGB = [
    [180, 180, 180],   # banda 0: calmarias  (cinza)
    [ 30, 100, 255],   # banda 1:  3–13 kt   (azul)
    [ 40, 255,  40],   # banda 2: 13–20 kt   (verde)
    [255, 255,   0],   # banda 3: 20–25 kt   (amarelo)
    [255, 150,   0],   # banda 4: 25–40 kt   (laranja)
    [255,  30,   0],   # banda 5:   40+ kt   (vermelho)
]

# ─── VÍDEO E GIF ──────────────────────────────────────────────────────────────
# Graus de rotação:
#   video_spin_deg = 180  →  pista gira 180° (uma cabeceira até a outra)
#   gif_spin_deg   = 360  →  pista dá volta completa no GIF
VIDEO_SPIN_DEG = 180
GIF_SPIN_DEG   = 360

# FPS do vídeo MP4.
# Duração ≈ (VIDEO_SPIN_DEG + 30) / FPS_VIDEO
# Exemplos: fps=14 → ~15s | fps=21 → ~10s | fps=7 → ~30s
FPS_VIDEO = 14

# Fator de aceleração do GIF em relação ao vídeo.
# GIF_SPEED_MULT=2 → GIF dura metade do tempo do vídeo (~7s)
# GIF_SPEED_MULT=1 → mesma velocidade que o vídeo (~15s)
GIF_SPEED_MULT = 2

print("✅ Parâmetros definidos na memória — execute a próxima célula para salvar.")
print(f"   Pista: {RUNWAY_LENGTH_M} m  |  FPS: {FPS_VIDEO}  |  GIF ×{GIF_SPEED_MULT}")
print(f"   Duração estimada do vídeo: {(VIDEO_SPIN_DEG + 30) / FPS_VIDEO:.1f}s")
print(f"   Duração estimada do GIF:   {(GIF_SPIN_DEG   + 30) / (FPS_VIDEO * GIF_SPEED_MULT):.1f}s")


In [ ]:
# ═══════════════════════════════════════════════════════
# 💾 SALVAR config_runway.json
# Execute esta célula após editar os parâmetros acima.
# ═══════════════════════════════════════════════════════
import json, os

CONFIG_PATH = os.path.join(PROJECT_ROOT, "config_runway.json")

nova_config = {
    "pista": {
        "_note": "Comprimento de pista em metros. O limite de vento cruzado e calculado automaticamente (RBAC154).",
        "_crosswind_por_comprimento": {
            ">=1500m": "20 kt  (37 km/h)",
            "1200-1500m": "13 kt  (24 km/h)",
            "<1200m": "10 kt  (19 km/h)"
        },
        "runway_length_m": RUNWAY_LENGTH_M
    },
    "rosa_dos_ventos": {
        "_note_bands": "Limites das bandas de velocidade em nos. Gera faixas: [0-3] [3-13] [13-20] [20-25] [25-40] [40+].",
        "wind_speed_bands_kts": WIND_BANDS_KTS,
        "_note_cores": "Formato [R, G, B] 0-255. Uma cor por banda (mesma ordem). Indice 0=[0-3kt], 1=[3-13], 2=[13-20], 3=[20-25], 4=[25-40], 5=[40+].",
        "cores_rgb": WIND_COLORS_RGB
    },
    "video": {
        "_note_spin": "Graus de rotacao da pista. video_spin_deg=180 (MP4 180 graus); gif_spin_deg=360 (GIF completo).",
        "video_spin_deg": VIDEO_SPIN_DEG,
        "gif_spin_deg":   GIF_SPIN_DEG,
        "_note_fps": "fps_video: quadros/s do MP4. gif_speed_multiplier: GIF N× mais rapido que o video.",
        "fps_video":          FPS_VIDEO,
        "gif_speed_multiplier": GIF_SPEED_MULT,
        "_note_cores": "Cores do video/GIF separadas da rosa para permitir paletas distintas.",
        "cores_rgb": VIDEO_COLORS_RGB
    },
    "localizacao": {
        "_note": "Opcional. Se informados, substituem as coordenadas lidas do cabecalho do CSV. Use null para desativar.",
        "latitude":  LATITUDE,
        "longitude": LONGITUDE
    },
    "declinacao_magnetica": {
        "_note": "Opcional. Se 'valor' for informado (graus; positivo=Leste, negativo=Oeste), a consulta a NOAA e ignorada. Use null para calcular automaticamente.",
        "valor": DECLINATION
    }
}

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(nova_config, f, ensure_ascii=False, indent=2)

# Recarrega a config do pipeline com os novos valores
cfg.load_runway_config()

# Tabela de limites de crosswind
lim = cfg.wind.crosswind_limit_kts
print("✅ config_runway.json salvo com sucesso!\n")
print(f"{'Parâmetro':<35} {'Valor':>15}")
print("─" * 52)
print(f"{'Comprimento da pista':<35} {RUNWAY_LENGTH_M:>12} m")
print(f"{'Limite de crosswind (RBAC154)':<35} {lim:>11.0f} kt")
print(f"{'Bandas de velocidade (nós)':<35} {str(WIND_BANDS_KTS):>15}")
print(f"{'FPS do vídeo MP4':<35} {FPS_VIDEO:>15}")
print(f"{'Multiplicador de velocidade GIF':<35} {GIF_SPEED_MULT:>14}×")
print(f"{'Duração estimada MP4':<35} {(VIDEO_SPIN_DEG+30)/FPS_VIDEO:>13.1f} s")
print(f"{'Duração estimada GIF':<35} {(GIF_SPIN_DEG+30)/(FPS_VIDEO*GIF_SPEED_MULT):>13.1f} s")
print(f"{'Rotação no MP4':<35} {VIDEO_SPIN_DEG:>14}°")
print(f"{'Rotação no GIF':<35} {GIF_SPIN_DEG:>14}°")
if LATITUDE:
    print(f"{'Latitude':<35} {LATITUDE:>15.6f}°")
    print(f"{'Longitude':<35} {LONGITUDE:>15.6f}°")
dec_str = f"{DECLINATION}°" if DECLINATION is not None else "Consulta automática NOAA"
print(f"{'Declinação magnética':<35} {dec_str:>15}")


In [ ]:
---
## 📂 SEÇÃO 3 — Dados de Entrada

Verifica os arquivos CSV em `data/raw/` e exibe estatísticas básicas.

> ⚠️ Coloque os arquivos `.csv` em `data/raw/` antes de continuar.


In [ ]:
# ─── Listar arquivos de entrada ────────────────────────────────────────────────
import glob as _glob

raw_dir   = cfg.output.data_raw
csv_files = sorted(_glob.glob(os.path.join(raw_dir, "*.csv")) +
                   _glob.glob(os.path.join(raw_dir, "*.CSV")))

if not csv_files:
    print(f"⚠️  Nenhum CSV encontrado em: {raw_dir}")
    print("    Copie os arquivos .csv para data/raw/ e execute novamente.")
else:
    print(f"✅ {len(csv_files)} arquivo(s) encontrado(s) em data/raw/\n")
    print(f"  {'Arquivo':<45} {'Tamanho':>10}")
    print("  " + "─" * 57)
    for f in csv_files:
        kb = os.path.getsize(f) / 1024
        print(f"  {os.path.basename(f):<45} {kb:>8.1f} KB")


In [ ]:
# ─── Pré-vizualização dos CSVs (primeiras linhas) ─────────────────────────────
import pandas as pd, chardet

for filepath in csv_files[:3]:   # mostra no máximo 3 arquivos
    with open(filepath, "rb") as fh:
        enc = chardet.detect(fh.read(50_000)).get("encoding", "utf-8")
    try:
        df_prev = pd.read_csv(filepath, encoding=enc, sep=None, engine="python", nrows=5)
        print(f"\n📄 {os.path.basename(filepath)}  (encoding={enc})")
        print(f"   Colunas: {list(df_prev.columns)}")
        display(df_prev)
    except Exception as e:
        print(f"⚠️  Erro ao ler {os.path.basename(filepath)}: {e}")


In [ ]:
---
## 🚀 SEÇÃO 4 — Pipeline Completo (Modo Rápido)

Executa **todos os estágios** de uma vez. Ideal para uso cotidiano após configurar.

- Limpa cache automaticamente antes de executar
- Gera: MP4, GIF 360°, PNG windrose e FinalResult.json

> ⏱️ Tempo estimado: 2–5 minutos (dependendo do volume de dados e se precisa consultar NOAA)


In [ ]:
import shutil, time

# ─── Limpeza do cache de saídas anteriores ────────────────────────────────────
exports_dir = os.path.join(cfg.output.data_gold, "exports")
if os.path.exists(exports_dir):
    shutil.rmtree(exports_dir)
    print("🗑️  Cache de exports limpo.")

# ─── Execução completa do pipeline ────────────────────────────────────────────
print("▶️  Iniciando pipeline HOROv1...\n")
t_start = time.time()

result = subprocess.run(
    [sys.executable, "orchestrator.py", "--all"],
    cwd=str(PROJECT_ROOT),
    capture_output=False,  # mostra o log em tempo real no terminal
    text=True
)

elapsed = time.time() - t_start
mins, secs = divmod(int(elapsed), 60)

if result.returncode == 0:
    print(f"\n✅ Pipeline concluído em {mins}m {secs}s")
    final_json = os.path.join(cfg.output.data_gold, "FinalResult.json")
    if os.path.exists(final_json):
        with open(final_json, encoding="utf-8") as f:
            final = json.load(f)
        print(f"\n📊 Resultados ({len(final)} estação/estações):")
        print("─" * 60)
        for station, by_years in final.items():
            for years, res in by_years.items():
                pct = res.get("fo_pct", 0)
                rwy = res.get("runway", "—")
                obs = res.get("total_observations", 0)
                print(f"  ✈️  {station}  |  {years} anos  |  Melhor pista: {rwy}  |  FO: {pct:.1f}%  ({obs:,} obs)")
else:
    print(f"\n❌ Pipeline terminou com erro (código {result.returncode})")
    print("   Verifique o log acima ou em pipeline_run.log")


In [ ]:
---
## 🔬 SEÇÃO 5 — Pipeline Estágio a Estágio (Modo Debug)

Execute cada estágio individualmente para inspecionar o estado intermediário.  
Útil para diagnosticar problemas ou explorar os dados em cada camada.

> ⚠️ Execute as células **em ordem**. Cada estágio depende do resultado do anterior.


In [ ]:
# ─── 5.0 Inicializar contexto do pipeline ─────────────────────────────────────
# Recarrega config e cria contexto limpo para os estágios
cfg.load_runway_config()
cfg.ensure_dirs()

input_files = cfg.output.input_csvs()
if not input_files:
    print("⚠️  Nenhum CSV em data/raw/. Adicione os arquivos antes de continuar.")
else:
    context = PipelineContext(input_files=input_files)
    print(f"✅ Contexto criado — run_id: {context.run_id}")
    print(f"   Arquivos de entrada: {len(input_files)}")
    for f in input_files:
        print(f"   • {os.path.basename(f)}")


In [ ]:
# ─── S0 — Merge Raw (opcional: mescla múltiplos CSVs da mesma estação) ────────
from pipeline.stages import s00_merge_raw
context = s00_merge_raw.run(context, cfg)
print("✅ S0 merge_raw concluído.")


In [ ]:
# ─── S1 — Ingest (Raw → Bronze) ───────────────────────────────────────────────
from pipeline.stages import s01_ingest
context = s01_ingest.run(context, cfg)

print("\n📦 Bronze:")
print(f"  {'Estação':<30} {'Status':<12} {'Linhas':>8}")
print("  " + "─" * 52)
for station, rec in context.bronze.items():
    status = "❌ REJEIT." if rec.rejected else "✅ OK"
    linhas = rec.row_count
    print(f"  {station:<30} {status:<12} {linhas:>8,}")
    if rec.rejected:
        print(f"    ⚠️  {rec.rejection_reason}")


In [ ]:
# ─── S2 — Validate ────────────────────────────────────────────────────────────
from pipeline.stages import s02_validate
context = s02_validate.run(context, cfg)
print("✅ S2 validate concluído.")


In [ ]:
# ─── S3 — Transform (Bronze → Silver) ────────────────────────────────────────
from pipeline.stages import s03_transform
context = s03_transform.run(context, cfg)

print("\n🥈 Silver:")
print(f"  {'Estação':<30} {'Registros':>10}  {'Período'}")
print("  " + "─" * 65)
for station, rec in context.silver.items():
    df = rec.df
    if df.empty:
        print(f"  {station:<30}    (vazio)")
        continue
    min_d = df["timestamp"].min().strftime("%Y-%m")
    max_d = df["timestamp"].max().strftime("%Y-%m")
    print(f"  {station:<30} {len(df):>10,}  {min_d} → {max_d}")
    print(f"  {'  Speed (kt)  min/mean/max':<35} {df['speed_kts'].min():.1f} / {df['speed_kts'].mean():.1f} / {df['speed_kts'].max():.1f}")


In [ ]:
# ─── S4 — Analyze (Tabelas de vento) ──────────────────────────────────────────
from pipeline.stages import s04_analyze
context = s04_analyze.run(context, cfg)

print("\n📊 Wind Tables:")
print(f"  {'Estação':<28} {'Janela':>8}  {'Observações':>13}  {'Calmas%':>8}")
print("  " + "─" * 62)
for station, by_years in context.wind_tables.items():
    for years, wt in by_years.items():
        print(f"  {station:<28} {years:>5} anos  {wt.total_records:>13,}  {wt.calm_pct:>7.1f}%")


In [ ]:
# ─── S5 — Enrich (Declinação magnética) ───────────────────────────────────────
# ⚠️ Requer Google Chrome instalado, ou defina DECLINATION no config acima.
from pipeline.stages import s05_enrich
context = s05_enrich.run(context, cfg)

print("\n🧭 Declinações magnéticas:")
print(f"  {'Estação':<35} {'Declinação':>12}")
print("  " + "─" * 50)
for station, rec in context.silver.items():
    dec = getattr(rec, "magnetic_declination", None)
    dec_str = f"{dec:.4f}°" if dec is not None else "N/A"
    print(f"  {station:<35} {dec_str:>12}")


In [ ]:
# ─── S6 — Optimize (Melhor orientação + renderização de frames) ───────────────
from pipeline.stages import s06_optimize
context = s06_optimize.run(context, cfg)

print("\n🏆 Resultados de otimização:")
print(f"  {'Estação':<28} {'Anos':>5}  {'Pista':>8}  {'FO%':>7}  {'Cross%':>7}  {'Rumo°':>6}")
print("  " + "─" * 70)
for station, by_years in context.results.items():
    for years, res in by_years.items():
        print(f"  {station:<28} {years:>5}  {res.runway_designation:>8}  "
              f"{res.fo_pct:>6.2f}%  {res.crosswind_pct:>6.2f}%  {res.best_heading_deg:>5.0f}°")


In [ ]:
# ─── S7 — Export (MP4 + GIF + PNG + JSON) ─────────────────────────────────────
from pipeline.stages import s07_export
context = s07_export.run(context, cfg)

print(f"\n✅ Export concluído! Estágios: {context.stages_executed}")
print(f"\n📁 Saídas em: {os.path.join(cfg.output.data_gold, 'exports')}")

# Lista os arquivos gerados
exports_root = os.path.join(cfg.output.data_gold, "exports")
for root, dirs, files in os.walk(exports_root):
    for fname in sorted(files):
        if fname.endswith((".mp4", ".gif", ".png", ".json")):
            fpath = os.path.join(root, fname)
            kb = os.path.getsize(fpath) / 1024
            rel = os.path.relpath(fpath, str(PROJECT_ROOT))
            print(f"  📄 {rel}  ({kb:.0f} KB)")


---
## 📈 SEÇÃO 6 — Resultados e Visualização

Exibe os arquivos gerados inline no notebook:
- Rosa dos ventos (PNG)
- Resultado numérico final (JSON)
- Abre MP4 e GIF no explorador de arquivos


In [ ]:
# ─── Exibir rosa dos ventos (PNG) inline para cada estação ────────────────────
from IPython.display import Image as IPImage
import glob as _glob

png_files = sorted(_glob.glob(
    os.path.join(cfg.output.data_gold, "exports", "**", "Windrose-*.png"),
    recursive=True
))

if not png_files:
    print("⚠️  Nenhum PNG de windrose encontrado. Execute o pipeline primeiro.")
else:
    for png in png_files:
        rel = os.path.relpath(png, str(PROJECT_ROOT))
        station = os.path.basename(os.path.dirname(png))
        print(f"\n🌹 {station}  →  {rel}")
        try:
            display(IPImage(filename=png, width=800))
        except Exception as e:
            print(f"   (não foi possível exibir inline: {e})")


In [ ]:
# ─── Exibir FinalResult.json ──────────────────────────────────────────────────
final_json_path = os.path.join(cfg.output.data_gold, "FinalResult.json")

if not os.path.exists(final_json_path):
    print("⚠️  FinalResult.json não encontrado. Execute o pipeline primeiro.")
else:
    with open(final_json_path, encoding="utf-8") as f:
        final = json.load(f)

    print("📋 FinalResult.json\n")
    for station, by_years in final.items():
        print(f"  🛫  Estação: {station}")
        for years_str, res in by_years.items():
            print(f"    ├─ Janela: {years_str} anos")
            print(f"    │  Melhor pista     : {res.get('runway', '—')}")
            print(f"    │  Fator de Operação: {res.get('fo_pct', 0):.2f}%")
            print(f"    │  Vento cruzado    : {res.get('crosswind_pct', 0):.2f}%")
            print(f"    │  Rumo magnético   : {res.get('best_heading_deg', 0):.0f}°")
            print(f"    └─ Total observações: {res.get('total_observations', 0):,}")
        print()


In [ ]:
# ─── Abrir MP4 e GIF no explorador de arquivos do Windows ─────────────────────
import subprocess as _sp

mp4_files = sorted(_glob.glob(
    os.path.join(cfg.output.data_gold, "exports", "**", "RunwayOrientation-*.mp4"),
    recursive=True
))
gif_files = sorted(_glob.glob(
    os.path.join(cfg.output.data_gold, "exports", "**", "RunwayOrientation-*.gif"),
    recursive=True
))

for mp4 in mp4_files:
    station = os.path.basename(os.path.dirname(mp4))
    size_mb = os.path.getsize(mp4) / 1024 / 1024
    print(f"🎬 Abrindo MP4: {station}  ({size_mb:.1f} MB)")
    _sp.Popen(["explorer", os.path.normpath(mp4)])

for gif in gif_files:
    station = os.path.basename(os.path.dirname(gif))
    size_mb = os.path.getsize(gif) / 1024 / 1024
    print(f"🎞️  Abrindo GIF: {station}  ({size_mb:.1f} MB)")
    _sp.Popen(["explorer", os.path.normpath(gif)])

if not mp4_files and not gif_files:
    print("⚠️  Nenhum MP4 ou GIF encontrado. Execute o pipeline primeiro.")
